# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Metadata object provides structured access:
print(f"Dataset loaded: {dataset.metadata.name}\nDescription: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available Record Sets and their @id
record_sets = dataset.metadata.record_sets
print(f"Total record sets found: {len(record_sets)}\n")
for idx, record_set in enumerate(record_sets):
    print(f"Record Set {idx+1} @id: {record_set.id}")
    print(f"  Name: {getattr(record_set, 'name', None)}")
    print(f"  Description: {getattr(record_set, 'description', None)}")
    print("  Fields and their @id:")
    for field in getattr(record_set, 'fields', []):
        print(f"    - {field.id} (name: {getattr(field, 'name', None)})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets into dataframes using their @id
all_record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in all_record_set_ids:
    # Load as DataFrame
    df = pd.DataFrame(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set {record_set_id}")

# For illustration, pick the first record set for further steps
if all_record_set_ids:
    first_record_set = all_record_set_ids[0]
    print(f"\nColumns in record set '{first_record_set}':\n{dataframes[first_record_set].columns.tolist()}")
    dataframes[first_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Select a numeric field from the first record set
# Please change the field_id to a numeric field's @id from the Data Overview output
record_set_id = first_record_set
df = dataframes[record_set_id]

# List available fields for this record set
print('Available columns (see their @id in section 2):')
print(df.columns.tolist())

# Try to select a likely numeric field (e.g., mass, abundance, PeptideCount, etc.)
# Replace the following with a real @id as appropriate for your dataset
possible_numeric_fields = [col for col in df.columns if any(x in col.lower() for x in ['count', 'abundance', 'mass', 'peptide', 'weight', 'coverage', 'mw'])]

print(f"Possible numeric fields for analysis: {possible_numeric_fields}")

if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]  # Use one as example
else:
    # Fall back to first column
    numeric_field_id = df.columns[0]

# Ensure numeric
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].mean()  # Use the mean as a threshold example
filtered_df = df[df[numeric_field_id] > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a key field (e.g., if a 'accession', 'sample', or 'description' column exists)
group_field = next((col for col in df.columns if any(x in col.lower() for x in ['accession', 'sample', 'description', 'type', 'group'])), None)
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Histogram for the numeric field
plt.figure(figsize=(7,4))
df[numeric_field_id].hist(bins=30)
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.title(f'Histogram of {numeric_field_id}')
plt.show()

# If grouping field available, boxplot by group
if group_field:
    plt.figure(figsize=(10,4))
    filtered_df.boxplot(column=numeric_field_id, by=group_field, grid=False, rot=45)
    plt.title(f'{numeric_field_id} distribution by {group_field}')
    plt.suptitle("")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.show()

# Pairplot if there are several numeric columns
numeric_cols = df.select_dtypes(include='number').columns
if len(numeric_cols) > 1:
    import seaborn as sns
    sns.pairplot(df[numeric_cols].sample(min(200, len(df))))
    plt.suptitle('Pairplot of numeric fields')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the Mass Spectrometry Analysis of Extracellular Vesicles dataset using its Croissant schema via `mlcroissant`, explored its metadata, record sets, and fields (referenced by their `@id`), extracted and analyzed data, and visualized important numeric features. The analysis pipeline is fully traceable and reproducible given the dataset's persistent and FAIR-compliant schema. 

Next steps could include specific domain-driven analyses, advanced feature engineering, or integrating the dataset with other proteomics FAIR datasets.